In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Dataset

# Setup Path
try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- PATCH FP16 ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
import math
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_thp_expdecay import THPExpDecay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")


In [ ]:
class HawkesGenerator:
    def __init__(self, mu, alpha, beta):
        """
        mu: Base intensity [M]
        alpha: Excitation matrix [M, M] (influence of j on i)
        beta: Decay matrix [M, M]
        """
        self.mu = np.array(mu)
        self.alpha = np.array(alpha)
        self.beta = np.array(beta)
        self.num_types = len(mu)
        
    def get_intensity(self, t, history):
        """Calcula lambda(t) dado histórico (t_i, k_i)."""
        # Base
        lam = self.mu.copy()
        
        # Excitation
        for t_i, k_i in history:
            if t_i >= t: break
            dt = t - t_i
            # Alpha coluna k_i (quem causou) -> influencia todas as linhas (quem sofre)
            # Beta idem
            decay = np.exp(-self.beta[:, k_i] * dt)
            lam += self.alpha[:, k_i] * decay
            
        return lam

    def simulate(self, T_max=100):
        """Ogata's Thinning Algorithm"""
        history = [] # (time, type)
        t = 0
        
        while t < T_max:
            # 1. Upper bound lambda_bar (intensidade atual é o máximo, pois decai)
            lam_curr = self.get_intensity(t, history)
            lam_bar = np.sum(lam_curr)
            
            # 2. Propose next time
            u = np.random.uniform(0, 1)
            w = -np.log(u) / lam_bar
            t += w
            
            if t >= T_max: break
            
            # 3. Accept/Reject
            lam_t = self.get_intensity(t, history)
            D = np.random.uniform(0, 1)
            
            if D * lam_bar <= np.sum(lam_t):
                # Accept - choose type
                # Probability proportional to intensity of each type
                lam_sum = np.sum(lam_t)
                probs = lam_t / lam_sum
                k = np.random.choice(self.num_types, p=probs)
                history.append((t, k))
                
        return history

    def generate_dataset(self, num_seqs, T_max=50, min_len=10):
        data = []
        print(f"Gerando {num_seqs} sequências Hawkes...")
        for _ in range(num_seqs):
            seq = []
            while len(seq) < min_len:
                seq = self.simulate(T_max)
            
            times = [x[0] for x in seq]
            types = [x[1] for x in seq]
            
            # Calcular deltas
            deltas = [times[0]] + [times[i]-times[i-1] for i in range(1, len(times))]
            
            data.append({
                'time_since_start': times,
                'time_since_last_event': deltas,
                'type_event': types
            })
        return data

# --- Configuração dos Cenários ---

# 1. Univariado (Auto-excitação forte)
gen_1d = HawkesGenerator(mu=[0.2], alpha=[[0.8]], beta=[[2.0]])

# 2. Multivariado (3D - Ciclo A->B->C->A)
# alpha[i, j]: j causa i
mu_3d = [0.1, 0.1, 0.1]
alpha_3d = [
    [0.2, 0.0, 0.6], # 0 excitado por 0(pouco) e 2(muito)
    [0.6, 0.2, 0.0], # 1 excitado por 0(muito) e 1(pouco)
    [0.0, 0.6, 0.2]  # 2 excitado por 1(muito) e 2(pouco)
]
beta_3d = [[3.0]*3 for _ in range(3)] # Decaimento rápido
gen_3d = HawkesGenerator(mu_3d, alpha_3d, beta_3d)


In [ ]:
class DictDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn_opt(batch_list, pad_id):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    
    # Cortar sequências muito longas para economizar memória (opcional)
    max_len = min(max_len, 200)
    
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    
    for i, item in enumerate(batch_list):
        l = min(len(item['time_since_start']), max_len)
        ts = torch.tensor(item['time_since_start'][:l], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'][:l], dtype=torch.float64)
        ev = torch.tensor(item['type_event'][:l], dtype=torch.long)
        
        # Sem normalização temporal aqui, pois os dados já estão em escala O(1)
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.0 # Sem dropout em sintético para overfitar bem
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()


In [ ]:
def compute_final_metrics(model, loader):
    model.eval()
    total_acc = 0
    total_rmse = 0
    total_nll = 0
    total_events = 0
    
    with torch.no_grad():
        for batch in loader:
            batch_gpu = tuple(t.to(device) for t in batch)
            
            with torch.amp.autocast('cuda'):
                 loss, num_events = model.loglike_loss(batch_gpu)
            total_nll += loss.item()
            
            _, time_delta_target, type_target, mask_target, _ = batch_gpu
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch_gpu)
            
            target_types = type_target[:, 1:]
            target_deltas = time_delta_target[:, 1:]
            target_mask = mask_target[:, 1:]
            
            correct = (types_pred == target_types) * target_mask
            total_acc += correct.sum().item()
            se = ((dtimes_pred - target_deltas) ** 2) * target_mask
            total_rmse += se.sum().item()
            total_events += target_mask.sum().item()
            
    avg_nll = total_nll / (total_events + 1e-9)
    avg_acc = total_acc / (total_events + 1e-9)
    avg_rmse = np.sqrt(total_rmse / (total_events + 1e-9))
    return avg_nll, avg_acc, avg_rmse

def train_and_evaluate(generator, name_scenario, num_types):
    print(f"\n{'='*40}\nCENÁRIO: {name_scenario}\n{'='*40}")
    
    # 1. Gerar Dados
    train_raw = generator.generate_dataset(2000)
    test_raw = generator.generate_dataset(200)
    
    pad_id = num_types
    collate = lambda x: collate_fn_opt(x, pad_id)
    train_loader = DataLoader(DictDataset(train_raw), batch_size=64, shuffle=True, collate_fn=collate, num_workers=0)
    test_loader = DataLoader(DictDataset(test_raw), batch_size=64, shuffle=False, collate_fn=collate, num_workers=0)
    
    # 2. Treinar Modelos
    models = {}
    for model_name, cls in [('THP', THP), ('THP Decay', THPExpDecay)]:
        print(f">>> Treinando {model_name}...")
        config = ModelConfig(num_types, pad_id)
        model = cls(config).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
        
        for epoch in range(15): # 15 épocas deve bastar para sintético limpo
            model.train()
            total_loss = 0
            for batch in train_loader:
                batch = [t.to(device) for t in batch]
                optimizer.zero_grad()
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            if epoch % 5 == 0: print(f"  Ep {epoch}: Loss {total_loss:.4f}")
        models[model_name] = model
    
    # 3. Métricas
    results = []
    print(f"\n--- {name_scenario} Métricas Finais ---")
    for name, model in models.items():
        nll, acc, rmse = compute_final_metrics(model, test_loader)
        results.append({'Model': name, 'NLL': nll, 'Acc': acc, 'RMSE': rmse})
    print(pd.DataFrame(results))
    
    # 4. Visualizar Ground Truth vs Predição
    sample = test_raw[0]
    batch = collate([sample])
    pad_time, pad_delta, pad_type, _, attn = [t.to(device) for t in batch]
    
    t_seq = pad_time[0].cpu().numpy()
    types = pad_type[0].cpu().numpy()
    valid_len = len(sample['time_since_start'])
    t_seq = t_seq[:valid_len]
    history = list(zip(t_seq, types[:valid_len]))
    
    for k in range(num_types):
        fig, ax = plt.subplots(figsize=(12, 5))
        t_start = t_seq[0]
        t_end = t_seq[min(20, valid_len-1)]
        t_grid = np.linspace(t_start, t_end, 500)
        
        lam_gt = []
        for t in t_grid:
            lam_gt.append(generator.get_intensity(t, history)[k])
        ax.plot(t_grid, lam_gt, 'k-', label='Ground Truth (Hawkes)', linewidth=2, alpha=0.6)
        
        colors = {'THP': 'gray', 'THP Decay': 'orange'}
        linestyles = {'THP': '--', 'THP Decay': '-'}
        
        for name, model in models.items():
            t_plot = []
            l_plot = []
            model.eval()
            for i in range(min(20, valid_len-1)):
                t_prev = t_seq[i]
                t_next = t_seq[i+1]
                mask = (t_grid >= t_prev) & (t_grid <= t_next)
                if not mask.any(): continue
                ts_interval = torch.tensor(t_grid[mask] - t_prev, device=device).float().view(1, 1, -1)
                with torch.no_grad():
                    L = pad_time.shape[1]
                    sample_dtimes = torch.zeros(1, L, len(ts_interval[0,0]), device=device)
                    sample_dtimes[:, i, :] = ts_interval.squeeze()
                    lambdas = model.compute_intensities_at_sample_times(
                        pad_time, pad_delta, pad_type, sample_dtimes, attention_mask=attn
                    )
                    l_vals = lambdas[0, i, :, k].cpu().numpy()
                    t_plot.extend(t_grid[mask])
                    l_plot.extend(l_vals)
            ax.plot(t_plot, l_plot, color=colors[name], linestyle=linestyles[name], label=name, linewidth=2)

        event_times = [t for t, y in zip(t_seq, types) if y == k and t <= t_end]
        for et in event_times:
            ax.axvline(x=et, color='red', alpha=0.3, ymin=0, ymax=0.1)
        ax.set_title(f'Intensity Matching (Type {k}) - {name_scenario}')
        ax.legend()
        plt.show()

train_and_evaluate(gen_1d, "Univariado", 1)
train_and_evaluate(gen_3d, "Multivariado 3D", 3)
